# Week 11 · Notebook 2 — RAG Experiments
**CSCI 250 · Fall 2026**

Build intuition: change **chunk_size**, **chunk_overlap**, and **k**, watch retrieval quality shift, and confirm the system says **"I don't know"** for out-of-scope questions. Retrieval-only — **no API key required**.

## 0. Install

In [ ]:
!pip -q install langchain-text-splitters chromadb sentence-transformers

## 1. The document set

In [ ]:
COURSE_HANDBOOK = '''
Attendance policy: This is an online asynchronous course. Modules open Monday
and the weekly lab is due Sunday at 11:59 PM Pacific. You may miss deadlines
twice with no penalty using the two automatic 48-hour extensions.

AI-use policy: Students may use AI assistants such as Claude and Gemini on
assignments but must understand and disclose their use with an AI Use note.
Exams, including the midterm and final, are AI-restricted.

Grading: Labs and assignments are 50 percent, the midterm is 20 percent, and
the final project is 30 percent. The final project is due December 19.

Office hours: Tuesday 5 to 6 PM and Saturday 10 to 11 AM Pacific on Zoom.
Email the instructor with the subject prefix CSCI250 for a reply within 48 hours.
'''
print('handbook length:', len(COURSE_HANDBOOK), 'chars')

## 2. A helper: index with given chunk settings, then retrieve
We rebuild a fresh Chroma collection for each experiment so settings don't leak.

In [ ]:
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

client = chromadb.Client()
_counter = {'n': 0}

def index_and_retrieve(question, chunk_size, chunk_overlap, k):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = [c.strip() for c in splitter.split_text(COURSE_HANDBOOK)]
    _counter['n'] += 1
    name = f"exp_{_counter['n']}"
    col = client.create_collection(name)
    col.add(documents=chunks,
            ids=[f'{name}_c{i}' for i in range(len(chunks))])
    res = col.query(query_texts=[question], n_results=min(k, len(chunks)))
    return len(chunks), list(zip(res['documents'][0], res['distances'][0]))

## 3. Experiment A — chunk size
Same question, three chunk sizes. Small chunks are focused but may cut ideas; large chunks carry more context but retrieve noisier matches.

In [ ]:
Q = 'How many automatic extensions do I get and how long are they?'
for size in (120, 240, 600):
    n, hits = index_and_retrieve(Q, chunk_size=size, chunk_overlap=20, k=2)
    print(f'\nchunk_size={size}  -> {n} chunks')
    for doc, dist in hits:
        print(f'  dist={dist:.3f} | {doc[:80]}...')

## 4. Experiment B — overlap
Overlap repeats text across boundaries so a sentence that straddles a split isn't lost. Compare 0 vs 60 characters of overlap.

In [ ]:
for ov in (0, 60):
    n, hits = index_and_retrieve(Q, chunk_size=140, chunk_overlap=ov, k=2)
    print(f'\nchunk_overlap={ov}  -> {n} chunks')
    for doc, dist in hits:
        print(f'  dist={dist:.3f} | {doc[:80]}...')

## 5. Experiment C — top-k
More chunks give the model more to work with, but too many dilute the signal and cost tokens. Start with k = 3–5.

In [ ]:
for k in (1, 3, 5):
    n, hits = index_and_retrieve(Q, chunk_size=240, chunk_overlap=40, k=k)
    print(f'\nk={k}: retrieved {len(hits)} chunks')
    for doc, dist in hits:
        print(f'  dist={dist:.3f} | {doc[:70]}...')

## 6. Out-of-scope → "I don't know"
A good RAG system refuses when nothing relevant is retrieved. Distances for an off-topic question are large; we threshold on them. (Chroma returns squared L2 distance: smaller = more similar.)

In [ ]:
def is_answerable(question, threshold=1.2, k=3):
    n, hits = index_and_retrieve(question, 240, 40, k)
    best = min(d for _, d in hits)
    return best <= threshold, best

for q in ['When is the final project due?',
          'What is the airspeed velocity of an unladen swallow?']:
    ok, best = is_answerable(q)
    verdict = 'ANSWER' if ok else "I don't know (out of scope)"
    print(f'best_dist={best:.3f} -> {verdict}\n  Q: {q}\n')

## 7. Takeaways
- Most RAG quality comes from **retrieval** — tune chunking and `k` first.
- **Overlap** protects ideas that span chunk boundaries.
- A **distance threshold** turns "no good match" into an honest "I don't know" instead of a hallucination.
- Carry these settings (and the pipeline from Notebook 1) into your **Final Project**.